# Implement The BPE Tokenizer

The Target and Source are tokenized representations of the sequences. 

**Performs:**
1. The tokenizer maps every unique word or sub-word to a unique integer ID.
2. Adds special tokens.
   - &lt;SOS&gt; (Start of sentence): Tells the Decoder to start generating.
   - &lt;EOS&gt; (End of sentence): Tells the Decoder to stop generating.
   - &lt;UNK&gt; (Unknown)
   - &lt;PAD&gt; (Padding): Fills the remaining space in a batch so all sequences have the same length.
       - Example: seq_len = 4 "John ate &lt;PAD&gt; &lt;PAD&gt;" or "I am leaving &lt;PAD&gt;"
3. Converts texts to numerical representation.
   1. Example: "The rabbit" → [23, 14]

- "We trained on the **standard WMT 2014 English-German dataset** consisting of about 4.5 million sentence pairs. Sentences were encoded using **byte-pair encoding** [3], which has a shared source target vocabulary of about 37000 tokens. For **English-French**, we used the significantly **larger WMT 2014 English-French dataset** consisting of 36M sentences and split tokens into a 32000 **word-piece** vocabulary [38]. Sentence pairs were batched together by approximate **sequence length**. Each training batch contained a set of sentence pairs containing approximately 25000 source tokens and 25000 target tokens."
    - For English-German they used the byte-pair encoding (**BPE**) tokenizer.
    - For English-French they used **WordPiece** tokenizer.

**Byte-Pair Encoding Sub-Word Tokenizer**

Sub-word tokenizer breaks down words, e.g., "transformer" → [""trans", "former"]

Instead of building the tokenizer my self I use a `tokenizers` library.

In [1]:
from tokenizers import Tokenizer, decoders, pre_tokenizers, processors
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import os

In [ ]:
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from datasets import DatasetDict
    from ..configs import english_german_config


def build_and_train_BPE_tokenizer(
    cfg: english_german_config,
    dataset_iterator: DatasetDict,
    perc_to_download:int
):
    """
    Build and train the BPE tokenizer that the paper used for the standard WMT 2014 English-German dataset.

    Args:
        cfg: Adds the correct vocab_size_dim to it.
        perc_to_download: Percentage of the database to append to BPE tokenizer filename.
    """
    print("Initializing BPE tokenizer...")

    # Init BPE model
    tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

    # Set the Pre-Tokenizer. Turns "The rabbit" → ["_The", "_rabbit"]
    tokenizer.pre_tokenizer = (
        pre_tokenizers.Metaspace()
    )  # Paper used -> pre_tokenizers.Whitespace()

    # TODO fix issue with tokenizer being trained everytime, and if its trained on 1% of database, and I then use a larger database.

    # Tell the tokenizer how to merge sub-words back into words, e.g., ["rab", "bit"] → "rabbit"
    #   and ["_The", "_rabbit"] → "The rabbit"
    tokenizer.decoder = decoders.Metaspace()  # Paper used -> decoders.BPEDecoder()

    # Configure the Trainer
    trainer = BpeTrainer(
        vocab_size=cfg.vocab_size_constraint,
        special_tokens=["<PAD>", "<UNK>", "<SOS>", "<EOS>"],
        # Integer representations: <PAD> = 0, "<UNK>" = 1, "<SOS>" = 2,  "<EOS>" = 3
        show_progress=True,
    )

    # Train the tokenizer on the shared dataset
    tokenizer.train_from_iterator(dataset_iterator, trainer=trainer)

    # Save the tokenizer
    save_path = os.path.join(cfg.MODEL_DIR, "saved_models")
    file_name = f"wmt_14_shared_bpe_tokenizer_{perc_to_download}_ds_percent_.json"

    os.makedirs(save_path, exist_ok=True)
    tokenizer.save(f"{save_path}/{file_name}")
    print(f"Tokenizer saved to {save_path}/{file_name}")

    # Add the correct vocab_size_dim to config
    cfg.vocab_size_dim = tokenizer.get_vocab_size()

    # Add <SOS> and <EOS> tokens.
    tokenizer.post_processor = processors.TemplateProcessing(
        single="<SOS> $A <EOS>", # Is the sentence sequence tokens.
        special_tokens=[
            ("<SOS>", tokenizer.token_to_id("<SOS>")),
            ("<EOS>", tokenizer.token_to_id("<EOS>")),
        ],
    )

    return tokenizer

In [ ]:
def test():
    import sys
    import os

    if "model" in os.getcwd():
        sys.path.append(os.path.abspath(".."))  # We need to grab load_wmt14_en_de
    from configs import english_german_config
    from utils.load_wmt14_en_de_dataset import load_wmt14_en_de, get_training_corpus

    cfg = english_german_config()

    raw_ds = load_wmt14_en_de(save_path=cfg.DATA_DIR, perc_to_download=cfg.perc_to_download)
    tokenizer = build_and_train_BPE_tokenizer(
        cfg=cfg, dataset_iterator=get_training_corpus(raw_ds), perc_to_download=cfg.perc_to_download
    )

    test_sentence = "The brown rabbit ate the apple."
    print(f"\n\n\nTest sentence: {test_sentence}")
    
    # Tokenize
    tokenized = tokenizer.encode(test_sentence)
    print(f"Tokens: {tokenized.tokens}")
    print(f"IDs: {tokenized.ids}")

    # DeTokenize
    detokenize = tokenizer.decode(tokenized.ids)
    print(f"Detokenized: {detokenize}")


test()

/Users/tonyavis/miniconda3/envs/AI_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




Attempting to get dataset...
Dataset already downloaded, loading from: /Users/tonyavis/Main/AI_projects_and_res/Transformer/data/wmt14/de-en
successfully loaded dataset!

Initializing BPE tokenizer...



Tokenizer saved to /Users/tonyavis/Main/AI_projects_and_res/Transformer/model/saved_models/wmt_14_shared_bpe_tokenizer_1_ds_percent_.json



Test sentence: The brown rabbit ate the apple.
Tokens: ['<SOS>', '▁The', '▁bro', 'wn', '▁ra', 'b', 'bit', '▁a', 'te', '▁the', '▁app', 'le.', '<EOS>']
IDs: [2, 378, 2590, 1186, 839, 59, 12877, 131, 156, 142, 581, 12305, 3]
Detokenized: The brown rabbit ate the apple.
